# NLP Text Classification Pipeline (10,000+ Documents)

This notebook trains a text classification model on the **20 Newsgroups** dataset
(a classic, publicly available NLP benchmark bundled with scikit-learn, containing
**~18,800 newsgroup posts across 20 categories** — well over 10,000 records).

**Pipeline covered:**
1. Load data (auto-downloads from the internet on first run)
2. Explore the dataset
3. Clean & preprocess text (NLTK)
4. Train/test split
5. Feature extraction (TF-IDF)
6. Train & compare multiple models (Naive Bayes, Logistic Regression, Linear SVM)
7. Evaluate (accuracy, precision/recall/F1, confusion matrix)
8. Save the best model + vectorizer for reuse
9. Run inference on new/unseen text

> **Note:** Replacing the dataset — If you have your own CSV instead, skip to
> **Section 2B** and point `df = pd.read_csv("your_file.csv")` at your file. The
> rest of the pipeline (Sections 3 onward) works unchanged as long as you have a
> text column and a label column.


## 1. Install & Import Libraries

In [ ]:
# Uncomment if running in a fresh environment
# !pip install scikit-learn pandas numpy nltk matplotlib seaborn joblib -q

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
import joblib

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

pd.set_option("display.max_colwidth", 150)
print("Libraries loaded successfully.")


## 2. Load the Dataset (Online Source)

`fetch_20newsgroups` downloads the dataset automatically the first time it's run
(cached locally afterward), giving us **~18,846 documents** across 20 topic
categories — comfortably over the 10,000-row requirement.


In [ ]:
categories = None  # None = load all 20 categories. You can restrict to a subset, e.g.:
# categories = ["rec.sport.baseball", "sci.space", "talk.politics.misc", "comp.graphics"]

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),  # strip metadata noise for cleaner text
    shuffle=True,
    random_state=42,
)

df = pd.DataFrame({
    "text": newsgroups.data,
    "label": newsgroups.target
})
df["label_name"] = df["label"].apply(lambda i: newsgroups.target_names[i])

print(f"Total documents: {len(df):,}")
print(f"Number of classes: {df['label'].nunique()}")
df.head()


### 2B. (Optional) Use Your Own CSV Instead

If you have your own labeled dataset, comment out Section 2 above and use this instead:

```python
df = pd.read_csv("your_file.csv")   # must contain a text column and a label column
df = df.rename(columns={"your_text_col": "text", "your_label_col": "label_name"})
df = df.dropna(subset=["text", "label_name"]).reset_index(drop=True)
print(f"Total rows: {len(df):,}")
```


## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
plt.figure(figsize=(10, 6))
df["label_name"].value_counts().plot(kind="barh")
plt.title("Document Count per Category")
plt.xlabel("Number of Documents")
plt.tight_layout()
plt.show()

# Basic text length stats
df["char_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().apply(len)
print(df[["char_len", "word_count"]].describe())


In [ ]:
# Check for missing / empty documents
print("Missing text values:", df["text"].isna().sum())
print("Empty-string documents:", (df["text"].str.strip() == "").sum())

# Drop empty documents if any
df = df[df["text"].str.strip() != ""].reset_index(drop=True)
print(f"Documents remaining: {len(df):,}")


## 4. Text Preprocessing

Standard NLP cleaning steps:
- Lowercasing
- Removing punctuation, numbers, and extra whitespace
- Removing stopwords
- Lemmatization


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)          # URLs
    text = re.sub(r"\S+@\S+", " ", text)                  # emails
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)  # punctuation
    text = re.sub(r"\d+", " ", text)                       # numbers
    text = re.sub(r"\s+", " ", text).strip()               # extra whitespace

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens if tok not in stop_words and len(tok) > 2]
    return " ".join(tokens)

# Apply cleaning (this may take a minute or two on ~18k documents)
df["clean_text"] = df["text"].apply(clean_text)

# Drop rows that became empty after cleaning
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print(f"Documents after cleaning: {len(df):,}")
df[["text", "clean_text"]].head(3)


## 5. Train/Test Split

In [ ]:
X = df["clean_text"]
y = df["label_name"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train):,} documents")
print(f"Test set:     {len(X_test):,} documents")


## 6. Feature Extraction (TF-IDF)

Converts cleaned text into numerical vectors weighted by term importance.


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,   # cap vocabulary size
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=3,             # ignore terms in fewer than 3 documents
    max_df=0.9,           # ignore overly common terms
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test):  {X_test_tfidf.shape}")


## 7. Train & Compare Models

We train three common baselines for text classification and compare accuracy:
- **Multinomial Naive Bayes** — fast, strong baseline for text
- **Logistic Regression** — solid linear baseline, interpretable
- **Linear SVM** — often the best performer for high-dimensional sparse TF-IDF data


In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "Linear SVM": LinearSVC(),
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    results[name] = {"model": model, "preds": preds, "accuracy": acc}
    print(f"{name}: accuracy = {acc:.4f}")


In [ ]:
# Pick the best-performing model automatically
best_name = max(results, key=lambda k: results[k]["accuracy"])
best_model = results[best_name]["model"]
best_preds = results[best_name]["preds"]

print(f"Best model: {best_name} (accuracy = {results[best_name]['accuracy']:.4f})")


## 8. Evaluation

In [ ]:
print(f"=== Classification Report: {best_name} ===\n")
print(classification_report(y_test, best_preds))


In [ ]:
# Confusion matrix (top categories, for readability if there are many classes)
labels_order = sorted(y_test.unique())
cm = confusion_matrix(y_test, best_preds, labels=labels_order)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=labels_order, yticklabels=labels_order)
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Save the Trained Model & Vectorizer

Saved artifacts can be reloaded later without retraining.


In [ ]:
joblib.dump(best_model, "best_text_classifier.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")

print("Saved: best_text_classifier.joblib")
print("Saved: tfidf_vectorizer.joblib")


## 10. Inference on New Text

Load the saved model/vectorizer and classify new, unseen text.


In [ ]:
def predict_category(raw_text, model=best_model, vec=vectorizer):
    cleaned = clean_text(raw_text)
    vectorized = vec.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return prediction

sample_texts = [
    "The new graphics card significantly improves rendering performance in games.",
    "NASA announced a new mission to study the moons of Jupiter.",
    "The senator proposed a new bill regarding healthcare reform.",
]

for text in sample_texts:
    print(f"Text: {text}\nPredicted category: {predict_category(text)}\n")


## Next Steps / Extending This Notebook

- **More data:** Swap in your own CSV in Section 2B (works with any text + label columns, any size).
- **Deep learning:** Replace Section 6-7 with a Keras/TensorFlow model using an Embedding + LSTM/CNN layer over tokenized sequences.
- **Transformers:** Fine-tune a pretrained model (e.g. `distilbert-base-uncased`) via HuggingFace `transformers` + `datasets` for state-of-the-art accuracy, at the cost of more compute.
- **Hyperparameter tuning:** Use `GridSearchCV`/`RandomizedSearchCV` on the TF-IDF + model pipeline.
- **Class imbalance:** If your own dataset is imbalanced, consider `class_weight="balanced"` in Logistic Regression/SVM or resampling techniques.
